# Test to start local launcher for single agents using llama-agents

key-features of agent in llama-agents of llama-index
- each agent has its own running microservice
- control plane
  - routes and distributes tasks via customizable LLM
- message queue
  - pass messagess between agents using a message queue
- agentic orchestrator
  - decides which agents are relevant to the task

compoments
- control pane server
- message queue
  - single message queue
- agent service
- launcher
  - local launcher
  - server launcer
- function tool
- function calling agent worker
- callback message consumer

launching
- message queue server
- control plane server
- ```tool agent``` servers

## multi-agent system

In [1]:
import dotenv
dotenv.load_dotenv() # our .env file defines OPENAI_API_KEY
from llama_agents import (
    AgentService,
    ControlPlaneServer,
    SimpleMessageQueue,
    AgentOrchestrator,
    LocalLauncher,
)
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.tools import FunctionTool
from llama_index.llms.ollama import Ollama
from llama_index.llms.openai import OpenAI
import logging

# turn on logging so we can see the system working
logging.getLogger("llama_agents").setLevel(logging.INFO)

In [2]:
# ResponseError: tiger-gemma2 does not support tools
# llm = Ollama(
#     model='tiger-gemma2',
#     request_timeout=30000.0,
#     keep_alive="10m",
#     additional_kwargs={"mirostat": 0, "keep_alive": "10m"}
# )

import os
llm = OpenAI()

## set single message queue and control plane

In [3]:
# Set up the message queue and control plane
message_queue = SimpleMessageQueue()

control_plane = ControlPlaneServer(
    message_queue=message_queue,
    orchestrator=AgentOrchestrator(llm=llm),
    port=37000
)

## single local launcer 

In [4]:
# create a tool
def get_the_secret_fact() -> str:
    """Returns the secret fact."""
    return "The secret fact is: A baby llama is called a 'Cria'."

tool = FunctionTool.from_defaults(fn=get_the_secret_fact)

# Define an agent
worker = FunctionCallingAgentWorker.from_tools([tool], llm=OpenAI())
agent = worker.as_agent()

# Create an agent service
agent_service = AgentService(
    agent=agent,
    message_queue=message_queue,
    # description="Useful for getting the secret fact.",
    # service_name="secret_fact_agent",
    description="General purpose assistant",
    service_name="assistant",
)

launcher = LocalLauncher(
    [agent_service],
    control_plane,
    message_queue,
)

result = await launcher.alaunch_single("What's the secret fact?")
print(result)

INFO:llama_agents.message_queues.simple - Consumer AgentService-30fa1551-73ba-4592-ba90-79d89a580712: assistant has been registered.
INFO:llama_agents.message_queues.simple - Consumer dee9e6fe-d974-4fe7-95cc-1ff82e56ddf2: human has been registered.
INFO:llama_agents.message_queues.simple - Consumer ControlPlaneServer-d18208c5-64da-4f48-abb9-4f038b99c1c7: control_plane has been registered.
INFO:llama_agents.services.agent - assistant launch_local
INFO:llama_agents.message_queues.base - Publishing message to 'control_plane' with action 'ActionTypes.NEW_TASK'
INFO:llama_agents.message_queues.simple - Launching message queue locally
INFO:llama_agents.services.agent - Processing initiated.
INFO:llama_agents.message_queues.base - Publishing message to 'human' with action 'ActionTypes.COMPLETED_TASK'
INFO:llama_agents.message_queues.simple - Successfully published message 'control_plane' to consumer.
INFO:llama_agents.message_queues.simple - Successfully published message 'human' to consumer.

I'm not sure what secret fact you're referring to. Could you please provide more context or clarify your question?
